# COZI enrichment between annotated cell types

COZI, the conditional z-score of Schiller et al. (2025), measures how strongly one cell type sits next to another in a core. For a pair of cell types A and B, the conditional cells are the cells of type A with at least one neighbour of type B. The observed score is the number of A to B neighbour links per conditional cell. Permuting the neighbour labels 300 times gives the scores expected by chance, and the z-score compares the observed score with them. The score is directed: B around A differs from A around B.

`data/cozi_all_fovs.csv` holds COZI scores for every core, and `cozienrichment.ipynb` reads it, but the code that computed it is not in the repository. This notebook recomputes the scores with the COZI fork of scimap and compares them with that file. It writes `data/cozi_celltypes.csv`, which `multicellular/SpatialFeatureContribution.ipynb` and `EncodingClinicalInformation.ipynb` use, and leaves the old file untouched.

## Environment
The notebook runs in the `scimap_cozi` environment: Python 3.10 and the fork `SchapiroLabor/scimap_COZI`, installed from GitHub at a fixed commit, as the tutorial of `SchapiroLabor/NEP_scimap` describes. The fork needs three packages it does not declare: `natsort`, `mpmath`, and `setuptools` below version 81, which still ships `pkg_resources`. Packages installed for the user in `~/.local/lib/python3.10` would otherwise fill these gaps and shadow the environment, so the kernel sets `PYTHONNOUSERSITE=1`. `environments/README.md` describes how to create the environment and register the kernel, and `environments/scimap_cozi.yml` lists its versions.

In [1]:
import json
import os
import time
import warnings
import numpy as np
import pandas as pd
import anndata as ad
import scimap as sm

warnings.filterwarnings("ignore", category=FutureWarning)
data_dir = "../../data/"
output_path = data_dir + "cozi_celltypes.csv"
old_path = data_dir + "cozi_all_fovs.csv"

n_permutations = 300  # as in the old file, see section 4
min_cells = 1         # keep every cell in the neighbour graph, see section 2
undefined = 1e6       # z-scores beyond this come from a zero spread, see section 3

assert os.environ.get("PYTHONNOUSERSITE") == "1", "run with the scimap_cozi kernel, which hides user site packages"
dist_info = os.path.join(os.path.dirname(sm.__file__), "..", "scimap-2.2.11.dist-info", "direct_url.json")
source = json.load(open(dist_info))
print("scimap", sm.__version__ if hasattr(sm, "__version__") else "2.2.11", "from", source["url"],
      "at commit", source["vcs_info"]["commit_id"])
print("numpy", np.__version__, "| pandas", pd.__version__, "| anndata", ad.__version__)

Running SCIMAP  2.2.11


/home/loan/miniforge3/envs/scimap_cozi/lib/python3.10/site-packages/mpl_scatter_density/__init__.py:4: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.

/home/loan/miniforge3/envs/scimap_cozi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



scimap 2.2.11 from https://github.com/SchapiroLabor/scimap_COZI.git at commit 191a1b1501ac463fc619f521462aba264cb34470
numpy 1.26.4 | pandas 2.3.3 | anndata 0.10.9


## 1. Cells
All cores and all annotated cells of `cell_table_with_types_stage.csv`, with the 15 cell types of the `consensus` annotation, `Unclear` included, and the cell centroids.

In [2]:
cell_table = pd.read_csv(data_dir + "cell_table_with_types_stage.csv",
                         usecols=["fov", "consensus", "centroid-0", "centroid-1"], low_memory=False)
cell_table["consensus"] = cell_table.consensus.astype(str)
print(f"{len(cell_table)} cells in {cell_table.fov.nunique()} cores, {cell_table.consensus.nunique()} cell types")
cell_table.consensus.value_counts().rename("cells").to_frame()

579011 cells in 469 cores, 15 cell types


,cells
consensus,
Cancer_cell,262411
Unclear,90917
CAF,75961
Monocyte,29922
Endothelial_cell,29646
Neutrophil,19169
CD4_Tcell,15340
Other_immune_cell,14025
CD68_Macrophage,10775


## 2. COZI
The settings:
- neighbours from a Delaunay triangulation of the cell centroids of each core
- conditional normalisation, which makes the score COZI
- z-scores against 300 permutations of the neighbour labels
- every cell type, however rare

The fork's default would drop the cell types with fewer than 10 cells in a core before building the neighbour graph. Removing cells links their neighbours to each other, which changes the scores of every pair in that core. All cells therefore stay in the graph, and section 3 keeps the number of conditional cells behind each score, so later analyses can require a minimum instead. This also matches MISTy in `MistyLines.ipynb`, where mistyR models a cell type in a core whenever it varies there, so a single cell is enough.

The fork fixes the random seed of the permutations, so a rerun on the same cells gives the same scores.

In [3]:
adata = ad.AnnData(np.zeros((len(cell_table), 1)), obs=cell_table.set_axis(cell_table.index.astype(str)))
start = time.time()
sm.tl.spatial_interaction(adata, phenotype="consensus", x_coordinate="centroid-0", y_coordinate="centroid-1",
                          imageid="fov", method="delaunay", normalization="conditional", pval_method="zscore",
                          permutation=n_permutations, min_celltype_count=min_cells, verbose=False, label="cozi")
print(f"{(time.time() - start) / 60:.1f} minutes")

2.2 minutes


## 3. Export
One row per core and ordered pair of cell types, named as in the old file: the cell type at the centre, then its neighbour. A pair has no score when either cell type is absent from the core. Such pairs stay out of the table instead of receiving a score of 0.

Some scores are undefined. When the permutations never change the score of a pair, their spread is zero and the z-score has no meaning. The fork then divides by 10⁻¹⁰, which returns either exactly 0 or a value around 10¹⁰. Both are left out as well. A pair whose cell types are present but never neighbours keeps its score, which is negative, and has 0 conditional cells.

The table keeps the number of conditional cells behind each score, so later analyses can require a minimum.

In [4]:
result = adata.uns["cozi"].copy()
result["interaction"] = result.phenotype.astype(str) + "_" + result.neighbour_phenotype.astype(str)
result = result.drop(columns=["phenotype", "neighbour_phenotype"]).set_index("interaction")
columns = {"zscore_": "zscore", "pvalue_": "pvalue", "cond_cell_ratio_": "conditional cell ratio",
           "cond_cell_count_": "conditional cells"}
cozi = pd.concat({name: result.filter(like=prefix).rename(columns=lambda c: c.replace(prefix, ""))
                  .stack(future_stack=True) for prefix, name in columns.items()}, axis=1)
cozi = cozi.rename_axis(["interaction", "fov"]).reset_index().dropna(subset=["zscore"])
zero_spread = (cozi.zscore == 0) | (cozi.zscore.abs() > undefined)
print(f"{int(zero_spread.sum())} undefined scores from a zero spread are left out")
cozi = cozi[~zero_spread]
cozi[["conditional cell ratio", "conditional cells"]] = cozi[["conditional cell ratio", "conditional cells"]].fillna(0)
cozi = cozi[["fov", "interaction", "zscore", "pvalue", "conditional cell ratio", "conditional cells"]]
cozi = cozi.sort_values(["interaction", "fov"]).reset_index(drop=True)
assert np.isfinite(cozi.zscore).all()
cozi.to_csv(output_path, index=False)
print(f"{len(cozi)} scores for {cozi.fov.nunique()} cores and {cozi.interaction.nunique()} ordered pairs")
cozi.describe().round(2)

300 undefined scores from a zero spread are left out


77019 scores for 469 cores and 225 ordered pairs


,zscore,pvalue,conditional cell ratio,conditional cells
count,77019.00,77019.00,77019.00,77019.00
mean,1.07,0.37,0.25,18.47
std,5.82,0.34,0.30,66.65
min,-103.98,0.00,0.00,0.00
25%,-0.41,0.01,0.00,0.00
50%,0.39,0.31,0.12,3.00
75%,1.89,0.69,0.43,12.00
max,93.20,1.00,1.00,1928.00


## 4. Comparison with the old file
Three differences are known before looking at the numbers.
- The old file has a score for every pair in every core. Pairs without a score there are 0, while this notebook leaves them out.
- The old file is dated 27 May 2025. The fork as of then, commit `81cb913`, set the observed score of a pair to 0 when fewer than 5 conditional cells supported it, but left the permutations unchanged. Those pairs therefore received strongly negative z-scores, an avoidance that comes only from the low count. Later commits removed this step. With 300 permutations, that commit and the current one give identical scores for the pairs with at least 5 conditional cells, when run on the same cells.
- `MetabViz.ipynb` rewrote the cell table in January 2026, after the old file was made, so the input may differ.

The table splits the pairs of the old file by the number of conditional cells behind the new score. Pairs whose cell types never neighbour each other have no conditional cells, and both versions score them the same way, against an observed score of 0.

In [5]:
old = pd.read_csv(old_path, index_col=0).rename_axis("fov").reset_index().rename(columns={"value": "old zscore"})
merged = old.merge(cozi, on=["fov", "interaction"], how="left")
merged["support"] = np.select([merged.zscore.isna(), merged["conditional cells"] == 0, merged["conditional cells"] < 5],
                              ["no score in the new file", "never neighbours", "1 to 4 conditional cells"],
                              "5 or more conditional cells")

def compare(group):
    scored = group.dropna(subset=["zscore"])
    finite = scored[np.isfinite(scored["old zscore"])]
    return pd.Series({
        "pairs": len(group),
        "old score is 0": (group["old zscore"] == 0).mean(),
        "median old score": group["old zscore"].median(),
        "median new score": scored.zscore.median() if len(scored) else np.nan,
        "Pearson correlation": finite["old zscore"].corr(finite.zscore) if len(finite) > 2 else np.nan,
        "identical scores": np.isclose(finite["old zscore"], finite.zscore).mean() if len(finite) else np.nan,
    })

merged.groupby("support")[["old zscore", "zscore", "conditional cells"]].apply(compare).round(3)

,pairs,old score is 0,median old score,median new score,Pearson correlation,identical scores
support,,,,,,
1 to 4 conditional cells,24732.0,0.000,-1.617,0.886,0.058,0.000
5 or more conditional cells,32427.0,0.000,0.333,1.302,0.729,0.694
never neighbours,19860.0,0.000,-0.436,-0.436,1.000,1.000
no score in the new file,28506.0,0.999,0.000,NaN,NaN,NaN


Pairs whose cell types never neighbour each other score identically in both files. That confirms the settings of the old file: Delaunay neighbours, conditional normalisation and 300 permutations. Pairs without a score in the new file are 0 in the old file, because one of their cell types is absent from the core.

Pairs with 1 to 4 conditional cells score far lower in the old file, and the two files barely correlate there. This is the masking step of the old fork: it set the observed score of these pairs to 0, so the old file reports an avoidance that the cells do not show.

For pairs with at least 5 conditional cells, most scores are identical, but not all of them.

For the pairs with at least 5 conditional cells, both versions of the fork compute the same score, so differences there come from the input. The table counts, in each core, the share of these pairs whose score is identical in both files.

In [6]:
well_supported = merged[merged.support == "5 or more conditional cells"]
identical = well_supported.assign(identical=np.isclose(well_supported["old zscore"], well_supported.zscore))
per_core = identical.groupby("fov").identical.mean()
pd.Series({"cores": len(per_core), "cores with every score identical": int((per_core == 1).sum()),
           "cores with no score identical": int((per_core == 0).sum()),
           "median share identical per core": per_core.median(),
           "lowest share identical in a core": per_core.min()}).round(3).to_frame("value")

,value
cores,469.000
cores with every score identical,1.000
cores with no score identical,0.000
median share identical per core,0.689
lowest share identical in a core,0.387


Nearly every core has both identical and differing well-supported scores. The differences are therefore not confined to a few cores whose cells changed. They fit a rewrite of the cell table that touched cells in most cores, through their annotation or their order, which changes the permutations. This notebook cannot tell which.

## 5. Summary

The rerun reproduces the old file wherever the method and the cells are unchanged, so the settings are right. The old file has two problems, though. Pairs with 1 to 4 conditional cells carry the strongly negative scores of the old fork, an avoidance the cells do not support. Pairs with an absent cell type carry 0 instead of no score. Beyond these, part of the well-supported scores differ because the cell table changed after the old file was made.

`multicellular/SpatialFeatureContribution.ipynb` and `EncodingClinicalInformation.ipynb` use `data/cozi_celltypes.csv`. `cozienrichment.ipynb` still reads `data/cozi_all_fovs.csv`, so its results rest on the old scores. Both notebooks use every score, as MISTy uses every cell type present in a core. An analysis that needs well-supported scores can require a minimum number of conditional cells, which the table keeps.